In [1]:
# =============================================================================
# ACTIVIDAD 2 — Criterio 3: TimeGAN para series temporales con anomalías
# Curso: Inteligencia Artificial Generativa Aplicada al Análisis de Datos
# UNIR — Máster en Visual Analytics & Big Data
# =============================================================================
# DESCRIPCIÓN:
#   Serie temporal: temperatura horaria sintética con tendencia diaria y
#   estacionalidad intradiaria (sensor IoT de temperatura ambiente).
#   Pipeline: generación de datos → entrenamiento TimeGAN → generación de
#   200 trayectorias sintéticas → comparación visual y cuantitativa →
#   inyección de anomalías → evaluación de detección.
# =============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend sin pantalla para guardar PNG
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from statsmodels.tsa.stattools import acf, pacf
import os

# TensorFlow con manejo de errores robusto (igual que timegan_extendido.py)
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    tf.config.run_functions_eagerly(True)
except Exception as e:
    print(f"Error importando TensorFlow: {e}")
    raise

np.random.seed(42)
tf.random.set_seed(42)

OUTPUT_DIR = "outputs_timegan"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:

# =============================================================================
# SECCIÓN 1: GENERACIÓN DE DATOS REALES
# =============================================================================
# Serie temporal: temperatura horaria de un sensor IoT durante N días.
# Estructura:
#   - Componente circadiana: mínimo a las 6h (~12°C), máximo a las 15h (~28°C)
#   - Tendencia suave creciente (verano) a lo largo de los días
#   - Ruido gaussiano ligero (±1°C)
# Cada "serie" = una ventana de seq_len horas (un ciclo diario completo)
# =============================================================================

def generate_temperature_series(n_series=250, seq_len=24):
    """
    Genera n_series ciclos diarios de temperatura horaria (24h).
    Cada serie tiene:
      - Curva circadiana con mínimo nocturno y máximo vespertino
      - Ligera variación aleatoria de amplitud y fase entre días
      - Ruido gaussiano realista
    Returns: array de forma (n_series, seq_len, 1) normalizado [0, 1]
    """
    t = np.arange(seq_len)  # horas 0..23
    data = []

    for i in range(n_series):
        # Temperatura base: mínimo ~12°C a las 6h, máximo ~28°C a las 15h
        # Modelado como suma de dos sinusoides
        amp     = np.random.uniform(7.0, 10.0)       # amplitud del ciclo
        fase    = np.random.uniform(-0.3, 0.3)        # variación de fase entre días
        offset  = 20.0 + 0.05 * i                    # tendencia leve creciente (~+1°C cada 20 días)
        ruido   = np.random.randn(seq_len) * 0.8      # ruido sensor ±0.8°C

        # Curva circadiana: coseno desplazado al mínimo a las 6h
        temp = offset + amp * np.cos(2 * np.pi * (t - 15 + fase) / 24) + ruido
        data.append(temp.reshape(seq_len, 1))

    data = np.array(data, dtype=np.float32)  # (n_series, seq_len, 1)

    # Normalización min-max a [0, 1] para el entrenamiento de la GAN
    global TEMP_MIN, TEMP_MAX
    TEMP_MIN = data.min()
    TEMP_MAX = data.max()
    data_norm = (data - TEMP_MIN) / (TEMP_MAX - TEMP_MIN)

    return data_norm, data  # normalizada y original (°C)


def denormalize(data_norm):
    """Convierte de espacio [0,1] de vuelta a grados Celsius."""
    return data_norm * (TEMP_MAX - TEMP_MIN) + TEMP_MIN


print("=" * 65)
print("SECCIÓN 1: Generando serie temporal de temperatura horaria")
print("=" * 65)
TEMP_MIN, TEMP_MAX = 0.0, 1.0  # se sobreescribirán en generate_temperature_series
real_norm, real_celsius = generate_temperature_series(n_series=250, seq_len=24)
print(f"  Series generadas : {real_norm.shape[0]}")
print(f"  Longitud (horas) : {real_norm.shape[1]}")
print(f"  Rango real       : {real_celsius.min():.1f}°C – {real_celsius.max():.1f}°C")
print(f"  Media            : {real_celsius.mean():.2f}°C")
print(f"  Desv. estándar   : {real_celsius.std():.2f}°C")


SECCIÓN 1: Generando serie temporal de temperatura horaria
  Series generadas : 250
  Longitud (horas) : 24
  Rango real       : 9.2°C – 42.7°C
  Media            : 26.23°C
  Desv. estándar   : 7.00°C


In [3]:

# =============================================================================
# SECCIÓN 2: ARQUITECTURA TimeGAN
# =============================================================================
# Siguiendo la arquitectura del Tema 6:
#   Embedder  : serie real  → espacio latente (GRU encoder)
#   Recovery  : espacio latente → serie reconstruida (GRU decoder)
#   Generator : ruido Z → espacio latente sintético
#   Supervisor: latente sintético → latente supervisado (captura dinámica)
#   Discriminator: distingue latentes reales vs. sintéticos
# =============================================================================

def build_timegan_models(seq_len=24, dim=1, hidden_dim=32):
    """Construye los 5 componentes de TimeGAN con GRU."""

    # Embedder (encoder): serie real → representación latente
    X = layers.Input(shape=(seq_len, dim))
    h = layers.GRU(hidden_dim, return_sequences=True)(X)
    h = layers.Dense(hidden_dim, activation='tanh')(h)
    embedder = Model(X, h, name="embedder")

    # Recovery (decoder): representación latente → serie reconstruida
    H = layers.Input(shape=(seq_len, hidden_dim))
    x_tilde = layers.GRU(hidden_dim, return_sequences=True)(H)
    x_tilde = layers.Dense(dim, activation='sigmoid')(x_tilde)
    recovery = Model(H, x_tilde, name="recovery")

    # Generator: ruido gaussiano → espacio latente sintético
    Z = layers.Input(shape=(seq_len, dim))
    e = layers.GRU(hidden_dim, return_sequences=True)(Z)
    e = layers.Dense(hidden_dim, activation='tanh')(e)
    generator = Model(Z, e, name="generator")

    # Supervisor: captura la dinámica temporal en el espacio latente
    G_h = layers.Input(shape=(seq_len, hidden_dim))
    s = layers.GRU(hidden_dim, return_sequences=True)(G_h)
    s = layers.Dense(hidden_dim, activation='tanh')(s)
    supervisor = Model(G_h, s, name="supervisor")

    # Discriminator: clasifica latentes como reales (1) o sintéticos (0)
    H_in = layers.Input(shape=(seq_len, hidden_dim))
    y = layers.GRU(hidden_dim, return_sequences=False)(H_in)
    y = layers.Dense(1, activation='sigmoid')(y)
    discriminator = Model(H_in, y, name="discriminator")

    return embedder, recovery, generator, supervisor, discriminator


In [4]:

# =============================================================================
# SECCIÓN 3: ENTRENAMIENTO TimeGAN (3 fases)
# =============================================================================

def train_timegan(real_data, seq_len=24, dim=1,
                  epochs_ae=50, epochs_s=50, epochs_joint=150):
    """
    Entrena TimeGAN en 3 fases:
      Fase 1 (Autoencoder): embedder + recovery aprenden a reconstruir series.
      Fase 2 (Supervisor): aprende la dinámica temporal en el espacio latente.
      Fase 3 (Adversarial): entrenamiento conjunto GAN.
    Returns: datos sintéticos generados (misma forma que real_data).
    """
    embedder, recovery, generator, supervisor, discriminator = \
        build_timegan_models(seq_len, dim)

    opt_ae  = tf.keras.optimizers.Adam(0.001)
    opt_sup = tf.keras.optimizers.Adam(0.001)

    # ── FASE 1: Autoencoder ──────────────────────────────────────────────────
    print("\n  [FASE 1] Entrenando autoencoder (embedder + recovery)...")
    X_in = layers.Input(shape=(seq_len, dim))
    H    = embedder(X_in)
    X_tilde = recovery(H)
    ae = Model(X_in, X_tilde, name="autoencoder")
    ae.compile(optimizer=opt_ae, loss='mse')
    hist_ae = ae.fit(real_data, real_data,
                     epochs=epochs_ae, batch_size=32, verbose=0)
    print(f"     Loss final AE: {hist_ae.history['loss'][-1]:.5f}")

    # ── FASE 2: Supervisor ───────────────────────────────────────────────────
    print("  [FASE 2] Entrenando supervisor...")
    H_real = embedder.predict(real_data, verbose=0)
    supervisor.compile(optimizer=opt_sup, loss='mse')
    hist_sup = supervisor.fit(H_real, H_real,
                              epochs=epochs_s, batch_size=32, verbose=0)
    print(f"     Loss final SUP: {hist_sup.history['loss'][-1]:.5f}")

    # ── FASE 3: Adversarial ──────────────────────────────────────────────────
    print("  [FASE 3] Entrenamiento adversarial conjunto...")
    bce   = tf.keras.losses.BinaryCrossentropy()
    opt_d = tf.keras.optimizers.Adam(learning_rate=0.0002)
    opt_g = tf.keras.optimizers.Adam(learning_rate=0.0002)

    try:
        def train_step(real_batch):
            batch_size = tf.shape(real_batch)[0]
            Z = tf.random.normal(shape=(batch_size, seq_len, dim))

            with tf.GradientTape(persistent=True) as tape:
                # Camino real
                H_r    = embedder(real_batch, training=True)
                X_r    = recovery(H_r, training=True)
                # Camino sintético
                E_hat  = generator(Z, training=True)
                H_hat  = supervisor(E_hat, training=True)
                X_hat  = recovery(H_hat, training=True)
                # Discriminador
                y_real = discriminator(H_r, training=True)
                y_fake = discriminator(H_hat, training=True)
                # Pérdidas
                d_loss = (bce(tf.ones_like(y_real), y_real) +
                          bce(tf.zeros_like(y_fake), y_fake))
                g_adv  = bce(tf.ones_like(y_fake), y_fake)
                g_reco = tf.reduce_mean(tf.keras.losses.mse(real_batch, X_hat))
                g_loss = g_adv + 100.0 * g_reco

            d_vars = discriminator.trainable_variables
            g_vars = (generator.trainable_variables +
                      supervisor.trainable_variables +
                      recovery.trainable_variables)

            opt_d.apply_gradients(
                zip(tape.gradient(d_loss, d_vars), d_vars))
            opt_g.apply_gradients(
                zip(tape.gradient(g_loss, g_vars), g_vars))

            return d_loss, g_loss, g_reco

        dataset = (tf.data.Dataset
                   .from_tensor_slices(real_data)
                   .shuffle(500).batch(32))

        for epoch in range(epochs_joint):
            for batch in dataset:
                d_l, g_l, r_l = train_step(batch)
            if (epoch + 1) % 100 == 0:
                print(f"     Epoch {epoch+1:3d} | D_loss={d_l:.4f} "
                      f"| G_loss={g_l:.4f} | Reco={r_l:.5f}")

        # Generación de 200+ trayectorias sintéticas
        n_gen = max(200, real_data.shape[0])
        Z_gen = tf.random.normal(shape=(n_gen, seq_len, dim))
        E_hat = generator.predict(Z_gen, verbose=0)
        H_hat = supervisor.predict(E_hat, verbose=0)
        X_hat = recovery.predict(H_hat, verbose=0)
        print(f"  Trayectorias sintéticas generadas: {n_gen}")
        return X_hat, embedder, recovery, generator, supervisor

    except Exception as e:
        print(f"  Advertencia: entrenamiento adversarial falló ({e})")
        print("  Fallback: generación por perturbación del espacio latente")
        H_real = embedder.predict(real_data, verbose=0)
        noise  = 0.15 * np.random.randn(*H_real.shape).astype(np.float32)
        H_noisy = np.clip(H_real + noise, 0, 1)
        X_hat = recovery.predict(H_noisy, verbose=0)
        n_gen = max(200, real_data.shape[0])
        # Ampliar hasta 200 por repetición con ruido
        idx = np.random.choice(len(X_hat), size=n_gen, replace=True)
        X_hat = X_hat[idx] + 0.02 * np.random.randn(n_gen, seq_len, dim).astype(np.float32)
        X_hat = np.clip(X_hat, 0, 1)
        return X_hat.astype(np.float32), embedder, recovery, generator, supervisor


print("\n" + "=" * 65)
print("SECCIÓN 3: Entrenando TimeGAN")
print("=" * 65)
synthetic_norm, embedder, recovery, generator, supervisor = train_timegan(
    real_norm, seq_len=24, dim=1,
    epochs_ae=20, epochs_s=20, epochs_joint=1
)

# Desnormalizar para análisis en °C
synthetic_celsius = denormalize(synthetic_norm)
print(f"\n  Media sintética   : {synthetic_celsius.mean():.2f}°C")
print(f"  Std  sintética    : {synthetic_celsius.std():.2f}°C")







SECCIÓN 3: Entrenando TimeGAN

  [FASE 1] Entrenando autoencoder (embedder + recovery)...


C:\Users\purdi\anaconda3\envs\IA_GEN\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


     Loss final AE: 0.00080
  [FASE 2] Entrenando supervisor...
     Loss final SUP: 0.00006
  [FASE 3] Entrenamiento adversarial conjunto...
  Trayectorias sintéticas generadas: 250

  Media sintética   : 22.66°C
  Std  sintética    : 2.11°C


In [5]:
# =============================================================================
# SECCIÓN 4: COMPARACIÓN CUANTITATIVA Y VISUAL
# =============================================================================

print("\n" + "=" * 65)
print("SECCIÓN 4: Comparación real vs. sintético")
print("=" * 65)

# ── 4a: Métricas cuantitativas ───────────────────────────────────────────────
real_mean  = real_celsius.mean(axis=0).squeeze()
synth_mean = synthetic_celsius.mean(axis=0).squeeze()
real_std   = real_celsius.std(axis=0).squeeze()
synth_std  = synthetic_celsius.std(axis=0).squeeze()

print(f"  Media real global  : {real_celsius.mean():.3f}°C")
print(f"  Media sint. global : {synthetic_celsius.mean():.3f}°C")
print(f"  Std  real global   : {real_celsius.std():.3f}°C")
print(f"  Std  sint. global  : {synthetic_celsius.std():.3f}°C")
print(f"  Diferencia de medias: {abs(real_celsius.mean()-synthetic_celsius.mean()):.3f}°C")

# ── 4b: Gráfico 1 — Media ± STD a lo largo del día ──────────────────────────
horas = np.arange(24)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horas, real_mean, 'C0-o', ms=4, label='Real — media')
ax.fill_between(horas, real_mean - real_std, real_mean + real_std,
                color='C0', alpha=0.2, label='Real ± 1σ')
ax.plot(horas, synth_mean, 'C1--s', ms=4, label='Sintético — media')
ax.fill_between(horas, synth_mean - synth_std, synth_mean + synth_std,
                color='C1', alpha=0.2, label='Sintético ± 1σ')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Temperatura (°C)')
ax.set_title('Figura 1. Media ± STD horaria: datos reales vs. sintéticos (TimeGAN)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_mean_std.png", dpi=150)
plt.close()
print("  Figura 1 guardada: fig1_mean_std.png")

# ── 4c: Gráfico 2 — Ejemplos de trayectorias individuales ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, ax in enumerate(axes[0]):
    ax.plot(horas, real_celsius[i, :, 0], 'C0')
    ax.set_title(f'Real #{i+1}')
    ax.set_ylim(5, 40); ax.grid(alpha=0.3)
for i, ax in enumerate(axes[1]):
    ax.plot(horas, synthetic_celsius[i, :, 0], 'C1--')
    ax.set_title(f'Sintético #{i+1}')
    ax.set_ylim(5, 40); ax.grid(alpha=0.3)
for ax in axes.flat:
    ax.set_xlabel('Hora'); ax.set_ylabel('°C')
fig.suptitle('Figura 2. Ejemplos de trayectorias individuales: real (azul) vs. sintético (naranja)',
             fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_trayectorias.png", dpi=150)
plt.close()
print("  Figura 2 guardada: fig2_trayectorias.png")

# ── 4d: Gráfico 3 — Distribuciones marginales ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(real_celsius.flatten(), bins=50, alpha=0.6, color='C0',
        label='Real', density=True)
ax.hist(synthetic_celsius.flatten(), bins=50, alpha=0.6, color='C1',
        label='Sintético', density=True)
ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('Densidad')
ax.set_title('Figura 3. Distribución marginal de temperaturas: real vs. sintético')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_distribucion.png", dpi=150)
plt.close()
print("  Figura 3 guardada: fig3_distribucion.png")

# ── 4e: Gráfico 4 — ACF y PACF ──────────────────────────────────────────────
real_mean_series  = real_celsius.mean(axis=0).squeeze()
synth_mean_series = synthetic_celsius.mean(axis=0).squeeze()
nlags = 12

real_acf_vals  = acf(real_mean_series,  nlags=nlags, fft=False)
synth_acf_vals = acf(synth_mean_series, nlags=nlags, fft=False)
real_pacf_vals  = pacf(real_mean_series,  nlags=nlags)
synth_pacf_vals = pacf(synth_mean_series, nlags=nlags)

lags = np.arange(len(real_acf_vals))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ACF
ax = axes[0]
ax.stem(lags,       real_acf_vals,  linefmt='C0-', markerfmt='C0o',
        basefmt='k-', label='Real ACF')
ax.stem(lags + 0.2, synth_acf_vals, linefmt='C1--', markerfmt='C1s',
        basefmt='k-', label='Sintético ACF')
ax.set_title('Figura 4a. ACF — serie media (real vs. sintético)')
ax.set_xlabel('Lag (horas)'); ax.set_ylabel('ACF')
ax.legend(); ax.grid(alpha=0.3)

# PACF
ax = axes[1]
ax.stem(lags,       real_pacf_vals,  linefmt='C0-', markerfmt='C0o',
        basefmt='k-', label='Real PACF')
ax.stem(lags + 0.2, synth_pacf_vals, linefmt='C1--', markerfmt='C1s',
        basefmt='k-', label='Sintético PACF')
ax.set_title('Figura 4b. PACF — serie media (real vs. sintético)')
ax.set_xlabel('Lag (horas)'); ax.set_ylabel('PACF')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_acf_pacf.png", dpi=150)
plt.close()
print("  Figura 4 guardada: fig4_acf_pacf.png")

# ── 4f: Gráfico 5 — PCA 2D ──────────────────────────────────────────────────
n_r = real_celsius.shape[0]
n_s = synthetic_celsius.shape[0]
X_r = real_celsius.reshape(n_r, -1)
X_s = synthetic_celsius.reshape(n_s, -1)
X_all = np.vstack([X_r, X_s])
pca = PCA(n_components=2)
Z_all = pca.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(Z_all[:n_r, 0], Z_all[:n_r, 1], alpha=0.5, s=15,
           label=f'Real (n={n_r})', color='C0')
ax.scatter(Z_all[n_r:, 0], Z_all[n_r:, 1], alpha=0.5, s=15,
           label=f'Sintético (n={n_s})', color='C1')
ax.set_title('Figura 5. PCA 2D de trayectorias aplanadas (real vs. sintético)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_pca.png", dpi=150)
plt.close()
print("  Figura 5 guardada: fig5_pca.png")


SECCIÓN 4: Comparación real vs. sintético
  Media real global  : 26.227°C
  Media sint. global : 22.661°C
  Std  real global   : 7.002°C
  Std  sint. global  : 2.110°C
  Diferencia de medias: 3.566°C
  Figura 1 guardada: fig1_mean_std.png
  Figura 2 guardada: fig2_trayectorias.png
  Figura 3 guardada: fig3_distribucion.png
  Figura 4 guardada: fig4_acf_pacf.png
  Figura 5 guardada: fig5_pca.png


In [6]:
# =============================================================================
# SECCIÓN 4: COMPARACIÓN CUANTITATIVA Y VISUAL
# =============================================================================

print("\n" + "=" * 65)
print("SECCIÓN 4: Comparación real vs. sintético")
print("=" * 65)

# ── 4a: Métricas cuantitativas ───────────────────────────────────────────────
real_mean  = real_celsius.mean(axis=0).squeeze()
synth_mean = synthetic_celsius.mean(axis=0).squeeze()
real_std   = real_celsius.std(axis=0).squeeze()
synth_std  = synthetic_celsius.std(axis=0).squeeze()

print(f"  Media real global  : {real_celsius.mean():.3f}°C")
print(f"  Media sint. global : {synthetic_celsius.mean():.3f}°C")
print(f"  Std  real global   : {real_celsius.std():.3f}°C")
print(f"  Std  sint. global  : {synthetic_celsius.std():.3f}°C")
print(f"  Diferencia de medias: {abs(real_celsius.mean()-synthetic_celsius.mean()):.3f}°C")

# ── 4b: Gráfico 1 — Media ± STD a lo largo del día ──────────────────────────
horas = np.arange(24)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horas, real_mean, 'C0-o', ms=4, label='Real — media')
ax.fill_between(horas, real_mean - real_std, real_mean + real_std,
                color='C0', alpha=0.2, label='Real ± 1σ')
ax.plot(horas, synth_mean, 'C1--s', ms=4, label='Sintético — media')
ax.fill_between(horas, synth_mean - synth_std, synth_mean + synth_std,
                color='C1', alpha=0.2, label='Sintético ± 1σ')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Temperatura (°C)')
ax.set_title('Figura 1. Media ± STD horaria: datos reales vs. sintéticos (TimeGAN)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_mean_std.png", dpi=150)
plt.close()
print("  Figura 1 guardada: fig1_mean_std.png")

# ── 4c: Gráfico 2 — Ejemplos de trayectorias individuales ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, ax in enumerate(axes[0]):
    ax.plot(horas, real_celsius[i, :, 0], 'C0')
    ax.set_title(f'Real #{i+1}')
    ax.set_ylim(5, 40); ax.grid(alpha=0.3)
for i, ax in enumerate(axes[1]):
    ax.plot(horas, synthetic_celsius[i, :, 0], 'C1--')
    ax.set_title(f'Sintético #{i+1}')
    ax.set_ylim(5, 40); ax.grid(alpha=0.3)
for ax in axes.flat:
    ax.set_xlabel('Hora'); ax.set_ylabel('°C')
fig.suptitle('Figura 2. Ejemplos de trayectorias individuales: real (azul) vs. sintético (naranja)',
             fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_trayectorias.png", dpi=150)
plt.close()
print("  Figura 2 guardada: fig2_trayectorias.png")

# ── 4d: Gráfico 3 — Distribuciones marginales ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(real_celsius.flatten(), bins=50, alpha=0.6, color='C0',
        label='Real', density=True)
ax.hist(synthetic_celsius.flatten(), bins=50, alpha=0.6, color='C1',
        label='Sintético', density=True)
ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('Densidad')
ax.set_title('Figura 3. Distribución marginal de temperaturas: real vs. sintético')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_distribucion.png", dpi=150)
plt.close()
print("  Figura 3 guardada: fig3_distribucion.png")

# ── 4e: Gráfico 4 — ACF y PACF ──────────────────────────────────────────────
real_mean_series  = real_celsius.mean(axis=0).squeeze()
synth_mean_series = synthetic_celsius.mean(axis=0).squeeze()
nlags = 12

real_acf_vals  = acf(real_mean_series,  nlags=nlags, fft=False)
synth_acf_vals = acf(synth_mean_series, nlags=nlags, fft=False)
real_pacf_vals  = pacf(real_mean_series,  nlags=nlags)
synth_pacf_vals = pacf(synth_mean_series, nlags=nlags)

lags = np.arange(len(real_acf_vals))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ACF
ax = axes[0]
ax.stem(lags,       real_acf_vals,  linefmt='C0-', markerfmt='C0o',
        basefmt='k-', label='Real ACF')
ax.stem(lags + 0.2, synth_acf_vals, linefmt='C1--', markerfmt='C1s',
        basefmt='k-', label='Sintético ACF')
ax.set_title('Figura 4a. ACF — serie media (real vs. sintético)')
ax.set_xlabel('Lag (horas)'); ax.set_ylabel('ACF')
ax.legend(); ax.grid(alpha=0.3)

# PACF
ax = axes[1]
ax.stem(lags,       real_pacf_vals,  linefmt='C0-', markerfmt='C0o',
        basefmt='k-', label='Real PACF')
ax.stem(lags + 0.2, synth_pacf_vals, linefmt='C1--', markerfmt='C1s',
        basefmt='k-', label='Sintético PACF')
ax.set_title('Figura 4b. PACF — serie media (real vs. sintético)')
ax.set_xlabel('Lag (horas)'); ax.set_ylabel('PACF')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_acf_pacf.png", dpi=150)
plt.close()
print("  Figura 4 guardada: fig4_acf_pacf.png")

# ── 4f: Gráfico 5 — PCA 2D ──────────────────────────────────────────────────
n_r = real_celsius.shape[0]
n_s = synthetic_celsius.shape[0]
X_r = real_celsius.reshape(n_r, -1)
X_s = synthetic_celsius.reshape(n_s, -1)
X_all = np.vstack([X_r, X_s])
pca = PCA(n_components=2)
Z_all = pca.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(Z_all[:n_r, 0], Z_all[:n_r, 1], alpha=0.5, s=15,
           label=f'Real (n={n_r})', color='C0')
ax.scatter(Z_all[n_r:, 0], Z_all[n_r:, 1], alpha=0.5, s=15,
           label=f'Sintético (n={n_s})', color='C1')
ax.set_title('Figura 5. PCA 2D de trayectorias aplanadas (real vs. sintético)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig5_pca.png", dpi=150)
plt.close()
print("  Figura 5 guardada: fig5_pca.png")


SECCIÓN 4: Comparación real vs. sintético
  Media real global  : 26.227°C
  Media sint. global : 22.661°C
  Std  real global   : 7.002°C
  Std  sint. global  : 2.110°C
  Diferencia de medias: 3.566°C
  Figura 1 guardada: fig1_mean_std.png
  Figura 2 guardada: fig2_trayectorias.png
  Figura 3 guardada: fig3_distribucion.png
  Figura 4 guardada: fig4_acf_pacf.png
  Figura 5 guardada: fig5_pca.png


In [7]:
# =============================================================================
# SECCIÓN 5: INYECCIÓN DE ANOMALÍAS
# =============================================================================
# Se seleccionan 3 trayectorias reales y se inyectan 3 tipos de anomalías:
#   A1 - Pico puntual   : valor atípico único (+15°C a las 12h)
#   A2 - Cambio nivel   : salto estructural en la segunda mitad del día (+8°C)
#   A3 - Inversión ciclo: el ciclo diario se invierte (mínimo vespertino)
# =============================================================================

print("\n" + "=" * 65)
print("SECCIÓN 5: Inyección de anomalías")
print("=" * 65)

# Copias de las trayectorias reales originales (en °C)
anomaly_series = real_celsius[:3].copy()

# A1: pico puntual a las 12h
anomaly_series[0, 12, 0] += 15.0
print("  A1 inyectada: pico puntual +15°C a las 12h en serie 0")

# A2: cambio de nivel en la segunda mitad del día (horas 12-23)
anomaly_series[1, 12:, 0] += 8.0
print("  A2 inyectada: salto estructural +8°C horas 12-23 en serie 1")

# A3: inversión del ciclo (multiplicar la curva por -1 centrada en media)
media_serie = anomaly_series[2, :, 0].mean()
anomaly_series[2, :, 0] = 2 * media_serie - anomaly_series[2, :, 0]
print("  A3 inyectada: inversión del ciclo circadiano en serie 2")

# Normalizar las series anómalas al mismo espacio [0,1] del modelo
anomaly_norm = np.clip(
    (anomaly_series - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0, 1
).astype(np.float32)


SECCIÓN 5: Inyección de anomalías
  A1 inyectada: pico puntual +15°C a las 12h en serie 0
  A2 inyectada: salto estructural +8°C horas 12-23 en serie 1
  A3 inyectada: inversión del ciclo circadiano en serie 2


In [8]:

# =============================================================================
# SECCIÓN 6: EVALUACIÓN DE ANOMALÍAS CON TimeGAN
# =============================================================================
# Método: error de reconstrucción.
#   El modelo (embedder + recovery) reconstruye cada serie.
#   Series normales → error bajo (el modelo las conoce).
#   Series anómalas → error alto (el modelo no reconoce el patrón).
# Umbral: media + 2σ del error de reconstrucción sobre datos reales de test.
# =============================================================================

print("\n" + "=" * 65)
print("SECCIÓN 6: Evaluación de detección de anomalías")
print("=" * 65)

def reconstruction_error(series_norm, embedder, recovery):
    """Calcula el MSE de reconstrucción para cada serie."""
    H     = embedder.predict(series_norm, verbose=0)
    X_rec = recovery.predict(H, verbose=0)
    mse   = np.mean((series_norm - X_rec) ** 2, axis=(1, 2))
    return mse, X_rec

# Error en datos reales (referencia)
mse_real, _ = reconstruction_error(real_norm, embedder, recovery)
umbral = mse_real.mean() + 2 * mse_real.std()
print(f"  MSE real — media: {mse_real.mean():.6f} | std: {mse_real.std():.6f}")
print(f"  Umbral detección (media + 2σ): {umbral:.6f}")

# Error en series anómalas
mse_anom, rec_anom = reconstruction_error(anomaly_norm, embedder, recovery)
rec_anom_celsius = denormalize(rec_anom)

for i, (mse, tipo) in enumerate(zip(mse_anom,
        ["Pico puntual", "Cambio de nivel", "Inversión de ciclo"])):
    detectada = "✓ DETECTADA" if mse > umbral else "✗ No detectada"
    print(f"  A{i+1} ({tipo}): MSE={mse:.6f} → {detectada}")



SECCIÓN 6: Evaluación de detección de anomalías


C:\Users\purdi\anaconda3\envs\IA_GEN\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


  MSE real — media: 0.000928 | std: 0.000586
  Umbral detección (media + 2σ): 0.002099
  A1 (Pico puntual): MSE=0.005555 → ✓ DETECTADA
  A2 (Cambio de nivel): MSE=0.002845 → ✓ DETECTADA
  A3 (Inversión de ciclo): MSE=0.002026 → ✗ No detectada


In [9]:

# ── Gráfico 6 — Comparación MSE real vs. anómalo ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mse_real, bins=30, alpha=0.7, color='C0', label='MSE series reales', density=True)
ax.axvline(umbral, color='red', ls='--', lw=2, label=f'Umbral (μ+2σ) = {umbral:.5f}')
for i, mse in enumerate(mse_anom):
    ax.axvline(mse, color=f'C{i+2}', ls=':', lw=2,
               label=f'A{i+1}: MSE={mse:.5f}')
ax.set_xlabel('MSE de reconstrucción')
ax.set_ylabel('Densidad')
ax.set_title('Figura 6. Distribución MSE: series reales vs. anómalas\n'
             '(series anómalas con MSE > umbral son detectadas)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig6_mse_anomalias.png", dpi=150)
plt.close()
print("  Figura 6 guardada: fig6_mse_anomalias.png")

# ── Gráfico 7 — Trayectorias anómalas vs. reconstrucción ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titulos = ["A1: Pico puntual (+15°C, h=12)",
           "A2: Cambio nivel (+8°C, h≥12)",
           "A3: Inversión ciclo"]

for i, (ax, titulo) in enumerate(zip(axes, titulos)):
    ax.plot(horas, anomaly_series[i, :, 0], 'r-o', ms=4,
            label='Serie anómala')
    ax.plot(horas, rec_anom_celsius[i, :, 0], 'k--', lw=1.5,
            label='Reconstrucción TimeGAN')
    ax.fill_between(horas,
                    synth_mean - synth_std,
                    synth_mean + synth_std,
                    color='C1', alpha=0.15,
                    label='Rango sintético normal')
    ax.set_title(f'Figura 7{chr(97+i)}. {titulo}\nMSE={mse_anom[i]:.5f}',
                 fontsize=9)
    ax.set_xlabel('Hora'); ax.set_ylabel('°C')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Figura 7. Trayectorias anómalas vs. reconstrucción TimeGAN\n'
             '(la diferencia real–reconstruido revela la anomalía)', fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig7_anomalias_detalle.png", dpi=150)
plt.close()
print("  Figura 7 guardada: fig7_anomalias_detalle.png")

# =============================================================================
# RESUMEN FINAL
# =============================================================================
print("\n" + "=" * 65)
print("PIPELINE COMPLETADO")
print("=" * 65)
print(f"  Figuras guardadas en: {OUTPUT_DIR}/")
print(f"  Series reales        : {real_celsius.shape[0]}")
print(f"  Trayectorias sintéticas: {synthetic_celsius.shape[0]}")
print(f"  Anomalías inyectadas : 3 (pico, cambio nivel, inversión)")
print(f"  Umbral detección     : {umbral:.6f}")
print(f"  MSE anómalos         : {[round(float(m),6) for m in mse_anom]}")
detectadas = sum(1 for m in mse_anom if m > umbral)
print(f"  Anomalías detectadas : {detectadas}/3")

  Figura 6 guardada: fig6_mse_anomalias.png
  Figura 7 guardada: fig7_anomalias_detalle.png

PIPELINE COMPLETADO
  Figuras guardadas en: outputs_timegan/
  Series reales        : 250
  Trayectorias sintéticas: 250
  Anomalías inyectadas : 3 (pico, cambio nivel, inversión)
  Umbral detección     : 0.002099
  MSE anómalos         : [0.005555, 0.002845, 0.002026]
  Anomalías detectadas : 2/3
